# Трансформеры: механизм самовнимания
## Семинар 1

В этом семинаре мы шаг за шагом реализуем механизм самовнимания (self-attention)
в простейшей постановке: сначала в скалярной форме для небольшого числа векторов,
а затем в матричной форме. Мы будем:

- внимательно отслеживать размерности тензоров,
- сопоставлять код с формулами,
- интерпретировать полученные коэффициенты внимания и их влияние на выходные представления.

## Цели занятия

После выполнения этого семинара студент должен:

1. Понять, как механизм самовнимания сопоставляет входным векторам новые представления.
2. Освоить вычисление запросов, ключей и значений по линейным преобразованиям входов.
3. Научиться реализовывать softmax и взвешенные суммы векторных представлений.
4. Проанализировать, как меняются коэффициенты внимания при модификации входных данных.
5. Сравнить скалярную и матричную реализации самовнимания по вычислительной структуре.
6. Интерпретировать полученную матрицу внимания и обсуждать её свойства.

## 3. Теоретическая часть

### 3.1. Постановка задачи

Механизм самовнимания получает на вход последовательность векторов
$\mathbf{x}_1, \dots, \mathbf{x}_N$, где каждый вектор $\mathbf{x}_n \in \mathbb{R}^D$,
и возвращает последовательность векторов той же размерности
$\mathbf{x}'_1, \dots, \mathbf{x}'_N$, где $\mathbf{x}'_n \in \mathbb{R}^D$.

Идея: каждое новое представление $\mathbf{x}'_n$ является взвешенной суммой
значений (values) всех элементов последовательности, где веса определяются
скалярными произведениями запросов (queries) и ключей (keys).

### 3.2. Запросы, ключи и значения

Для каждого входного вектора $\mathbf{x}_n$ определяются три линейных
преобразования:

$$
\mathbf{q}_n = \Omega_q \mathbf{x}_n + \boldsymbol{\beta}_q, \quad
\mathbf{k}_n = \Omega_k \mathbf{x}_n + \boldsymbol{\beta}_k, \quad
\mathbf{v}_n = \Omega_v \mathbf{x}_n + \boldsymbol{\beta}_v,
$$

где $\Omega_q, \Omega_k, \Omega_v \in \mathbb{R}^{D \times D}$ — матрицы параметров,
а $\boldsymbol{\beta}_q, \boldsymbol{\beta}_k, \boldsymbol{\beta}_v \in \mathbb{R}^{D}$
— векторы смещений.

### 3.3. Коэффициенты внимания

Для фиксированного индекса $n$ (позиция, для которой строим выход) вычислим
«сырые» оценки внимания по скалярным произведениям запроса $\mathbf{q}_n$
со всеми ключами $\mathbf{k}_m$:

$$
 e_{nm} = \mathbf{q}_n^\top \mathbf{k}_m.
$$

Затем нормируем эти оценки с помощью softmax по индексу $m$:

$$
 \alpha_{nm} = \frac{\exp(e_{nm})}{\sum_{j=1}^N \exp(e_{nj})},
$$

так что для фиксированного $n$ выполняется $\sum_m \alpha_{nm} = 1$,
а все $\alpha_{nm} \ge 0$.

### 3.4. Взвешенная сумма значений

Новое представление $\mathbf{x}'_n$ вычисляется как
взвешенная сумма значений $\mathbf{v}_m$:

$$
 \mathbf{x}'_n = \sum_{m=1}^N \alpha_{nm} \, \mathbf{v}_m.
$$

Таким образом, каждый выходной вектор строится как комбинация всех входов,
где вклад каждого входа контролируется коэффициентами внимания.

Логика, которой будем следовать далее:

Теория → формулировка гипотезы → эксперимент в коде → интерпретация результата.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Настройки печати для удобства
np.set_printoptions(precision=3, suppress=True)

## 4. Практическая часть

### Задача 1. Генерация входных векторов и параметров

**Постановка задачи.**  
Сгенерировать небольшое количество входных векторов $\mathbf{x}_n$
и набор параметров $\Omega_q, \Omega_k, \Omega_v, \boldsymbol{\beta}_q,
\boldsymbol{\beta}_k, \boldsymbol{\beta}_v$.  
Зафиксировать `random seed` для воспроизводимости.

**Гипотеза.**  
При фиксированных параметрах инициализация входов определяет, какие элементы
последовательности будут «заметнее» для механизма внимания на старте.

**Теоретический минимум.**  
Размерность входного пространства обозначим $D$, а число векторов — $N$.
Каждый вектор будем хранить как столбец размерности `(D, 1)`.

In [ ]:
# Фиксируем зерно генератора случайных чисел для воспроизводимости
np.random.seed(3)

# Число входных векторов и их размерность
N = 3
D = 4

# Список входных векторов x_n формы (D, 1)
all_x = []
for n in range(N):
    x_n = np.random.normal(size=(D, 1))
    all_x.append(x_n)

print("Входные векторы x_n (список длины N):")
for i, x in enumerate(all_x):
    print(f"x_{i} shape = {x.shape}\n", x.T)

# Фиксируем отдельно для параметров
np.random.seed(0)

# Параметры для запросов, ключей и значений
omega_q = np.random.normal(size=(D, D))
omega_k = np.random.normal(size=(D, D))
omega_v = np.random.normal(size=(D, D))

beta_q = np.random.normal(size=(D, 1))
beta_k = np.random.normal(size=(D, 1))
beta_v = np.random.normal(size=(D, 1))

print("\nФормы матриц параметров:")
print("omega_q shape =", omega_q.shape)
print("omega_k shape =", omega_k.shape)
print("omega_v shape =", omega_v.shape)
print("beta_q shape  =", beta_q.shape)
print("beta_k shape  =", beta_k.shape)
print("beta_v shape  =", beta_v.shape)

### Задача 2. Вычисление запросов, ключей и значений

**Постановка задачи.**  
Реализовать линейные преобразования, которые переводят
входные векторы $\mathbf{x}_n$ в запросы, ключи и значения.

**Теоретический минимум.**  
Для каждого $n$:

$$
\mathbf{q}_n = \Omega_q \mathbf{x}_n + \boldsymbol{\beta}_q, \quad
\mathbf{k}_n = \Omega_k \mathbf{x}_n + \boldsymbol{\beta}_k, \quad
\mathbf{v}_n = \Omega_v \mathbf{x}_n + \boldsymbol{\beta}_v.
$$

Все результаты также имеют форму `(D, 1)`.

In [ ]:
# Вычисляем списки запросов, ключей и значений
all_queries = []
all_keys = []
all_values = []

for x in all_x:
    query = omega_q @ x + beta_q  # (D, 1)
    key   = omega_k @ x + beta_k  # (D, 1)
    value = omega_v @ x + beta_v  # (D, 1)

    all_queries.append(query)
    all_keys.append(key)
    all_values.append(value)

print("Формы q_n, k_n, v_n:")
for i, (q, k, v) in enumerate(zip(all_queries, all_keys, all_values)):
    print(f"n = {i}: q shape = {q.shape}, k shape = {k.shape}, v shape = {v.shape}")

### Задача 3. Реализация функции softmax

**Постановка задачи.**  
Реализовать функцию `softmax`, которая принимает список или массив
вещественных чисел и возвращает массив той же длины, где элементы
неотрицательны и суммируются к единице.

**Теоретический минимум.**  
Для вектора $\mathbf{z} = (z_1, \dots, z_M)$ softmax определяется как

$$
\mathrm{softmax}(z)_i = \frac{\exp(z_i)}{\sum_{j=1}^M \exp(z_j)}.
$$

Для численной устойчивости удобно вычитать максимум:

$$
\mathrm{softmax}(z)_i = \frac{\exp(z_i - z_{\max})}{\sum_{j} \exp(z_j - z_{\max})}.
$$

Гипотеза: сумма значений softmax всегда будет равна 1, а все элементы — неотрицательны.

In [ ]:
def softmax(scores):
    """Численно устойчивая реализация softmax для одномерного массива."""
    scores = np.asarray(scores, dtype=float)
    max_score = np.max(scores)
    exp_scores = np.exp(scores - max_score)
    denom = np.sum(exp_scores)
    return exp_scores / denom

# Пример проверки: сумма должна быть равна 1
example = np.array([1.0, 2.0, 3.0])
probs = softmax(example)
print("Входные значения:", example)
print("Softmax(probabilities):", probs)
print("Сумма вероятностей:", probs.sum())

### Задача 4. Скалярная реализация самовнимания

**Постановка задачи.**  
Для каждого выхода $\mathbf{x}'_n$ явно пройти по всем ключам $\mathbf{k}_m$,
вычислить скалярные произведения $e_{nm} = \mathbf{q}_n^\top \mathbf{k}_m$,
применить `softmax`, а затем построить взвешенную сумму значений
$\mathbf{v}_m$.

**Теоретический минимум.**  

$$
 e_{nm} = \mathbf{q}_n^\top \mathbf{k}_m, \quad
 \alpha_{nm} = \mathrm{softmax}(e_{n1}, \dots, e_{nN})_m, \quad
 \mathbf{x}'_n = \sum_{m=1}^N \alpha_{nm} \, \mathbf{v}_m.
$$

Гипотеза: для каждого выхода $n$ коэффициенты $\alpha_{nm}$ образуют
вероятностное распределение, а выходной вектор $\mathbf{x}'_n$ будет
сглаженной комбинацией значений $\mathbf{v}_m$.

In [ ]:
all_x_prime = []   # список выходных векторов x'_n формы (D, 1)
all_attentions = []  # коэффициенты внимания для каждого n

for n in range(N):
    q_n = all_queries[n]  # (D, 1)

    # Сырые скалярные произведения e_{nm}
    e_n = []
    for m in range(N):
        k_m = all_keys[m]  # (D, 1)
        dot_product = float(q_n.T @ k_m)  # скаляр
        e_n.append(dot_product)

    e_n = np.array(e_n)  # (N,)

    # Применяем softmax по m
    alpha_n = softmax(e_n)  # (N,)
    all_attentions.append(alpha_n)

    print(f"Коэффициенты внимания для выхода n={n}:")
    print(alpha_n, "сумма =", alpha_n.sum())

    # Взвешенная сумма значений v_m
    x_prime_n = np.zeros((D, 1))
    for m in range(N):
        v_m = all_values[m]  # (D, 1)
        x_prime_n += alpha_n[m] * v_m

    all_x_prime.append(x_prime_n)

print("\nПолученные выходные векторы x'_n:")
for i, x_p in enumerate(all_x_prime):
    print(f"x'_{i} shape = {x_p.shape}\n", x_p.T)

### Задача 5. Матричная реализация самовнимания и сравнение с поэлементной

**Постановка задачи.**  
Переписать вычисления самовнимания в матричной форме, используя
матрицу входов $X \in \mathbb{R}^{D \times N}$,
и сравнить результат с поэлементной реализацией.

**Теоретический минимум.**  

$$
X = [\mathbf{x}_1, \dots, \mathbf{x}_N] \in \mathbb{R}^{D \times N}.
$$

Тогда:

$$
Q = \Omega_q X + \boldsymbol{\beta}_q \mathbf{1}^\top,\quad
K = \Omega_k X + \boldsymbol{\beta}_k \mathbf{1}^\top,\quad
V = \Omega_v X + \boldsymbol{\beta}_v \mathbf{1}^\top,
$$

$$
E = Q^\top K \in \mathbb{R}^{N \times N},\quad
A = \mathrm{softmax}_{\text{по строкам}}(E),\quad
X' = V A^\top.
$$

Гипотеза: матричная реализация даст те же результаты, что и скалярная,
но в более компактной и эффективной форме.

In [ ]:
def softmax_rows(data_in: np.ndarray) -> np.ndarray:
    """Softmax по строкам матрицы data_in формы (N, N)."""
    max_per_row = np.max(data_in, axis=1, keepdims=True)
    exp_values = np.exp(data_in - max_per_row)
    denom = np.sum(exp_values, axis=1, keepdims=True)
    return exp_values / denom


def self_attention_matrix(all_x, omega_q, omega_k, omega_v, beta_q, beta_k, beta_v):
    """Матричная реализация самовнимания."""
    D = all_x[0].shape[0]
    N = len(all_x)

    # Собираем входы в матрицу X формы (D, N)
    X = np.zeros((D, N))
    for n, x in enumerate(all_x):
        X[:, n] = np.squeeze(x)

    ones_row = np.ones((1, N))  # (1, N)

    Q = omega_q @ X + beta_q @ ones_row  # (D, N)
    K = omega_k @ X + beta_k @ ones_row  # (D, N)
    V = omega_v @ X + beta_v @ ones_row  # (D, N)

    E = Q.T @ K  # (N, N)
    A = softmax_rows(E)  # (N, N)
    X_prime = V @ A.T  # (D, N)
    return X_prime, A


X_prime_mat, A_mat = self_attention_matrix(all_x, omega_q, omega_k, omega_v,
                                           beta_q, beta_k, beta_v)

print("Форма X_prime (матричная реализация):", X_prime_mat.shape)
print("Матрица вниманий A shape =", A_mat.shape)

# Собираем скалярный результат в матрицу для сравнения
X_prime_scalar = np.hstack(all_x_prime)  # (D, N)

print("\nСравнение столбцов X' (скалярная vs матричная):")
for n in range(N):
    print(f"n = {n}")
    print("scalar:", X_prime_scalar[:, n])
    print("matrix:", X_prime_mat[:, n])
    print()

In [ ]:
plt.figure(figsize=(4, 4))
plt.imshow(A_mat, cmap="viridis")
plt.colorbar(label="коэффициент внимания")
plt.xlabel("m (ключи)")
plt.ylabel("n (запросы)")
plt.title("Матрица вниманий A")
plt.tight_layout()
plt.show()

## 5. Интерпретация результатов

- Коэффициенты внимания для каждого выхода образуют распределение вероятностей
  по позициям входов: все значения неотрицательны и суммируются к единице.
- При фиксированных параметрах $\Omega_q, \Omega_k, \Omega_v$ и случайных входах
  некоторые позиции получают значительно больший вес, чем остальные, что приводит
  к доминированию соответствующих значений $\mathbf{v}_m$ во взвешенной сумме.
- Матричная реализация даёт те же результаты, что и скалярная, но позволяет
  записать вычисления в более компактной и эффективной форме.
- На поведение механизма влияет масштаб скалярных произведений запросов и ключей:
  чем больше по модулю элементы матрицы $E$, тем более «жёстким» становится
  распределение softmax (масса концентрируется на одном из элементов).

## 6. Выводы

1. Механизм самовнимания строит новые представления элементов последовательности
   как взвешенные суммы значений, где веса зависят от сходства запросов и ключей.
2. Линейные преобразования с параметрами $\Omega_q, \Omega_k, \Omega_v$ позволяют
   обучаемым образом задавать пространство, в котором измеряется «близость»
   между элементами.
3. Функция softmax обеспечивает интерпретируемость коэффициентов внимания
   как распределения вероятностей и усиливает наиболее важные связи.
4. Матричная формулировка самовнимания естественно выражается через операции
   линейной алгебры и хорошо масштабируется на длинные последовательности.
5. Численный анализ подчёркивает важность устойчивой реализации softmax
   и контроля масштабов скалярных произведений.

## 7. Самостоятельные задачи (с эталонными решениями)

### Задача 7.1. Влияние масштаба входного вектора

**Задача.**  
Скопируйте код скалярной реализации самовнимания и измените один из входных
векторов $\mathbf{x}_n$, умножив его на константу (например, 3 или 5).
Посмотрите, как изменятся коэффициенты внимания и выходные векторы.



**Ожидаемый результат.**  
Вес, соответствующий модифицированному вектору, как правило, возрастёт
(при прочих равных), так как изменятся скалярные произведения запросов
и ключей.

---

### Задача 7.2. Масштабирование скалярных произведений

**Задача.**  
Добавьте деление на $\sqrt{D}$ перед применением softmax:

$$
 e_{nm}^{(\text{scaled})} = \frac{\mathbf{q}_n^\top \mathbf{k}_m}{\sqrt{D}}.
$$

Сравните распределения внимания с немасштабированным случаем.



**Ожидаемый результат.**  
Распределения внимания становятся более «плавными»: веса менее экстремальные,
что уменьшает доминирование одной позиции.

---

### Задача 7.3. Альтернативная нормализация

**Задача.**  
Попробуйте заменить softmax на нормировку по $L^1$-норме без экспоненты
(просто взять абсолютные значения и поделить на сумму). Сравните поведение
с классическим softmax.

**Идея решения.**  
Реализовать альтернативную функцию нормализации и пересчитать внимания,
затем сравнить вектор $\mathbf{x}'_n$ и матрицу вниманий с исходными.